<a href="https://colab.research.google.com/github/taruchit/Building-RAG-for-answering-question-from-resume/blob/main/RAG_on_Resume.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
# Import the required packages
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

In [46]:
# Load the PDF as a LangChain document loader
loader = PyPDFLoader(file_path="/content/Resume_Taruchit.pdf")
resume_docs = loader.load()

In [47]:
# Inspect the first page
print(f"Total pages: {len(resume_docs)}")
print(resume_docs[0].page_content[:200])  # First 1000 characters
print(resume_docs[0].metadata)

Total pages: 2
TARUCHIT GOYAL 
Over 9.5 years of work experience in technology industry; 
Have worked with large scale distributed data platforms  
and data lake. Hands-on experience on building stable and 
efficien
{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-11-20T00:08:46+05:30', 'author': 'Taruchit Goyal', 'moddate': '2026-11-20T00:08:46+05:30', 'source': '/content/Resume_Taruchit.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


In [48]:
import os
from google.colab import userdata

# Securely retrieve the OpenAI API key from Colab Secrets
try:
    openai_api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    # Fallback to os.environ if not found in Colab Secrets
    openai_api_key = os.environ.get("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("Please set your 'OPENAI_API_KEY' in the Colab Secrets (the key icon on the left sidebar).")

# Load the models required to complete the exercise using the retrieved API key
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=openai_api_key)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=openai_api_key)

In [49]:
# Initialize RecursiveCharacterTextSplitter to make chunks of PDF text
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [50]:
# Split the resume document with text_splitter
splits = text_splitter.split_documents(resume_docs)

In [51]:
# Initialize Chroma vectorstore with documents as splits and using OpenAIEmbeddings
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

In [52]:
# Setup vectorstore as retriever
retriever = vectorstore.as_retriever()

In [53]:
# Define RAG prompt
prompt = ChatPromptTemplate.from_template("You are an assistant for question-answering tasks to hiring manager to give information about the candidate from the resume. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:")

In [54]:
# Setup the chain
rag_chain = (
    {"context": retriever , "question": RunnablePassthrough()}
    | prompt
    | llm
)

In [55]:
# Use case 1
query = "Does the candidate knows about GenAI?"
print("query: ",query)

query:  Does the candidate knows about GenAI?


In [56]:
# Invoke the query
answer = rag_chain.invoke(query).content
print(answer)

Yes, the candidate is knowledgeable about GenAI. They have worked on projects involving LLM models for translation and developed an AI-powered customer support agent using LangChain. Additionally, they designed a GAME framework for AI agents.


In [57]:
# Use case 2
query = "Does the candidate knows about C#?"
print("query: ",query)

query:  Does the candidate knows about C#?


In [58]:
# Invoke the query
answer = rag_chain.invoke(query).content
print(answer)

The context does not provide any information about the candidate's knowledge of C#. Therefore, I don't know if the candidate knows about C#.


In [59]:
# Use case 3
query = "What is the total years of experience for the candidate?"
print("query: ",query)

query:  What is the total years of experience for the candidate?


In [60]:
# Invoke the query
answer = rag_chain.invoke(query).content
print(answer)

The candidate has over 9.5 years of work experience in the technology industry.


In [61]:
# Use case 4
query = "Which all organizations has the candidate worked for?"
print("query: ",query)

query:  Which all organizations has the candidate worked for?


In [62]:
# Invoke the query
answer = rag_chain.invoke(query).content
print(answer)

The candidate has worked for State Street Corporation and Tata Consultancy Services. At State Street, they are currently a Data Engineer, and at Tata Consultancy Services, they served as a Splunk Developer from January 2017 to February 2022.
